# GPX1 Discovery Agent
## 01 — Model Development

Goal: establish a simple, interpretable molecular baseline before building any agentic behavior.

**Flow:** data audit → RDKit parsing → Morgan fingerprints → scaffold-aware split → activity model → screening enrichment.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from src.featurization import (
    parse_smiles,
    add_rdkit_descriptors,
    morgan_fingerprints,
)
from src.models import build_logistic_model
from src.campaign import scaffold_split
from src.evaluation import top_k_screening_table

RANDOM_SEED = 42
DATA_PATH = PROJECT_ROOT / "data" / "GPX1_curated_for_RL.csv"

### 1. Load and audit

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nActive rate:", f'{df["label"].mean():.3%}')
print("Unique scaffolds:", df["scaffold"].nunique())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate CIDs:", int(df["PUBCHEM_CID"].duplicated().sum()))
print("Duplicate SMILES:", int(df["PUBCHEM_EXT_DATASOURCE_SMILES"].duplicated().sum()))

With ~1.7% actives, accuracy is misleading: an always-inactive model would be ~98% accurate. Primary metrics are therefore **PR-AUC and screening enrichment**.

### 2. Parse chemistry and generate compact descriptors

In [ ]:
smiles_col = "PUBCHEM_EXT_DATASOURCE_SMILES"
mols, valid_mask = parse_smiles(df[smiles_col])

clean = df.loc[valid_mask].copy().reset_index(drop=True)
clean = add_rdkit_descriptors(clean, mols)

print(f"RDKit parsed {valid_mask.sum():,}/{len(valid_mask):,} SMILES")

clean.groupby("label")[
    ["MolWt", "LogP", "TPSA", "HBD", "HBA",
     "RotBonds", "RingCount", "FracCSP3"]
].median().T

### 3. Morgan fingerprints

In [ ]:
X = morgan_fingerprints(mols, radius=2, fp_size=2048)
y = clean["label"].to_numpy(dtype=int)
scaffolds = clean["scaffold"].astype(str).to_numpy()

print("Fingerprint matrix:", X.shape)

### 4. Scaffold-aware development/validation split

Holding out entire scaffolds reduces analogue leakage and provides a harder estimate of performance on unseen chemistry.

In [ ]:
train_idx, val_idx = scaffold_split(
    X, y, scaffolds,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

shared = set(scaffolds[train_idx]) & set(scaffolds[val_idx])

print("Train:", len(train_idx), "| actives:", int(y[train_idx].sum()))
print("Validation:", len(val_idx), "| actives:", int(y[val_idx].sum()))
print("Shared scaffolds:", len(shared))

### 5. Baseline model

In [ ]:
model = build_logistic_model(random_state=RANDOM_SEED)
model.fit(X[train_idx], y[train_idx])

p = model.predict_proba(X[val_idx])[:, 1]

roc_auc = roc_auc_score(y[val_idx], p)
pr_auc = average_precision_score(y[val_idx], p)
random_pr = y[val_idx].mean()

print("ROC-AUC:", round(roc_auc, 3))
print("PR-AUC:", round(pr_auc, 3))
print("Random PR baseline:", round(random_pr, 3))
print("PR-AUC / random:", round(pr_auc / random_pr, 1), "x")

### 6. Screening enrichment

In [ ]:
ranked = pd.DataFrame({
    "compound_index": val_idx,
    "PUBCHEM_CID": clean.iloc[val_idx]["PUBCHEM_CID"].to_numpy(),
    "true_label": y[val_idx],
    "predicted_active_score": p,
}).sort_values("predicted_active_score", ascending=False)

top_k_screening_table(
    ranked,
    budgets=[10, 20, 40, 100],
    active_rate=y[val_idx].mean(),
)

## Takeaway

A simple class-balanced logistic regression on Morgan fingerprints gives a strong scaffold-aware baseline and meaningful screening enrichment. The next notebook converts this model into a closed-loop experiment-selection system.